# 02 — PDE Basics: Types, Boundary Conditions & Initial Conditions

**Notebook Series**: Numerical Methods for PDEs  
**Prerequisites**: Calculus (partial derivatives), Linear Algebra, `numpy`, `matplotlib`

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Identify and classify PDEs (order, linearity, elliptic / parabolic / hyperbolic)
2. Understand what **Boundary Conditions (BCs)** are and when to use each type
3. Understand what **Initial Conditions (ICs)** are and how they combine with BCs
4. Visualise prototypical solutions of the three canonical PDE classes
5. Recognise well-posed vs. ill-posed problems

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
print('Libraries loaded successfully.')

---
## 1  What Is a PDE?

A **Partial Differential Equation (PDE)** is an equation relating a function of **two or more independent variables** to its partial derivatives.

General form for $u = u(x, y, \ldots, t)$:

$$F\!\left(x, y, t,\; u,\; \frac{\partial u}{\partial x},\; \frac{\partial u}{\partial y},\; \frac{\partial u}{\partial t},\; \frac{\partial^2 u}{\partial x^2},\; \ldots\right) = 0$$

### Key vocabulary

| Term | Definition | Example |
|------|------------|---------|
| **Order** | Highest derivative present | $u_{xx}$ → 2nd order |
| **Linear** | $u$ and its derivatives appear linearly (no products / powers) | $u_t = k\,u_{xx}$ |
| **Nonlinear** | Otherwise | $u_t + u\,u_x = 0$ (Burgers') |
| **Homogeneous** | Zero on the right-hand side | $u_{xx} + u_{yy} = 0$ |
| **Inhomogeneous** | Non-zero forcing term | $u_{xx} + u_{yy} = f(x,y)$ |

---
## 2  The Three Canonical Second-Order Linear PDEs

Every **linear second-order PDE** in two variables can be written as:

$$A\,u_{xx} + B\,u_{xy} + C\,u_{yy} + D\,u_x + E\,u_y + F\,u = G$$

The **discriminant** $\Delta = B^2 - 4AC$ determines its *character*:

| $\Delta$ | Type | Physical analogy |
|----------|------|------------------|
| $\Delta < 0$ | **Elliptic** | Steady-state / equilibrium |
| $\Delta = 0$ | **Parabolic** | Diffusion / heat flow |
| $\Delta > 0$ | **Hyperbolic** | Wave propagation |

### 2.1  Elliptic — Laplace / Poisson Equation

$$\nabla^2 u = u_{xx} + u_{yy} = 0 \quad \text{(Laplace)}$$
$$\nabla^2 u = f(x,y) \quad \text{(Poisson)}$$

- **No time dependence** — describes steady states  
- Physical examples: electrostatics, steady heat conduction, irrotational fluid flow  
- Requires BCs on the **entire boundary** of the domain

### 2.2  Parabolic — Heat / Diffusion Equation

$$u_t = \alpha\,u_{xx}$$

- One time-like direction → needs an **initial condition** + BCs on the spatial boundary  
- Physical examples: heat conduction, molecular diffusion, option pricing (Black-Scholes)  
- Solutions smooth out discontinuities instantaneously

### 2.3  Hyperbolic — Wave Equation

$$u_{tt} = c^2\,u_{xx}$$

- Two time-like directions → needs **two** initial conditions ($u$ and $u_t$ at $t=0$) + BCs  
- Physical examples: acoustics, electromagnetism, vibrating strings  
- Disturbances propagate at finite speed $c$; discontinuities are preserved

In [ ]:
# ── Visualise the discriminant idea ──────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
titles = ['Elliptic\n($\\Delta < 0$)\nLaplace: $u_{xx}+u_{yy}=0$',
          'Parabolic\n($\\Delta = 0$)\nHeat: $u_t = \\alpha u_{xx}$',
          'Hyperbolic\n($\\Delta > 0$)\nWave: $u_{tt} = c^2 u_{xx}$']

# --- Elliptic: steady 2-D solution u = sin(πx)sinh(πy)/sinh(π) on [0,1]² ---
x = np.linspace(0, 1, 100)
y = np.linspace(0, 1, 100)
X, Y = np.meshgrid(x, y)
U_elliptic = np.sin(np.pi * X) * np.sinh(np.pi * Y) / np.sinh(np.pi)
c0 = axes[0].contourf(X, Y, U_elliptic, 20, cmap='RdYlBu_r')
plt.colorbar(c0, ax=axes[0])
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')

# --- Parabolic: heat equation analytical solution -------------------------
x1 = np.linspace(0, 1, 200)
t_vals = [0.001, 0.01, 0.05, 0.1, 0.3]
alpha = 0.1
for t in t_vals:
    # Fourier series: u(x,t) = Σ b_n sin(nπx) exp(-α(nπ)²t)
    u = np.zeros_like(x1)
    for n in range(1, 60, 2):          # odd terms only for u₀=1
        bn = 4.0 / (n * np.pi)
        u += bn * np.sin(n * np.pi * x1) * np.exp(-alpha * (n * np.pi)**2 * t)
    axes[1].plot(x1, u, label=f't={t}')
axes[1].set_xlabel('x'); axes[1].set_ylabel('u(x,t)')
axes[1].legend(fontsize=8)

# --- Hyperbolic: d'Alembert wave solution ---------------------------------
x2 = np.linspace(0, 2, 300)
c = 1.0
# Initial condition: Gaussian pulse centred at x=1
f = lambda s: np.exp(-20 * (s - 1.0)**2)
for t in [0, 0.3, 0.6, 1.0]:
    u_wave = 0.5 * (f(x2 - c * t) + f(x2 + c * t))
    axes[2].plot(x2, u_wave, label=f't={t}')
axes[2].set_xlabel('x'); axes[2].set_ylabel('u(x,t)')
axes[2].legend(fontsize=8)

for ax, title in zip(axes, titles):
    ax.set_title(title, fontsize=10)

plt.suptitle('Three Canonical PDE Types — Typical Solutions', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 3  Boundary Conditions (BCs)

BCs specify what the solution does **on the boundary** $\partial\Omega$ of the spatial domain.  
Without them the problem has infinitely many solutions.

### 3.1  Dirichlet BC ("essential" or "first-kind")

The solution value is prescribed on the boundary:

$$u(x,t)\big|_{x \in \partial\Omega} = g(x,t)$$

**Physical meaning:** fixed temperature, fixed displacement, fixed voltage  
**Example:** a rod whose ends are held at 0 °C → $u(0,t)=0,\; u(L,t)=0$

### 3.2  Neumann BC ("natural" or "second-kind")

The **normal derivative** (flux) is prescribed:

$$\frac{\partial u}{\partial n}\bigg|_{\partial\Omega} = h(x,t)$$

**Physical meaning:** prescribed heat flux, stress, or flow rate; $h=0$ → insulated wall  
**Example:** insulated end of a rod → $u_x(L,t) = 0$

### 3.3  Robin BC ("mixed" or "third-kind")

A **linear combination** of value and flux:

$$\alpha\,u + \beta\,\frac{\partial u}{\partial n} = \gamma \quad \text{on } \partial\Omega$$

**Physical meaning:** Newton's cooling — heat loss proportional to the surface temperature  
**Example:** $u_x(L,t) + h\,u(L,t) = h\,T_{\infty}$ (convective cooling)

### 3.4  Periodic BC

$$u(0,t) = u(L,t), \qquad u_x(0,t) = u_x(L,t)$$

**Physical meaning:** a domain that wraps around (e.g., atmospheric models, ring geometry)

### 3.5  Summary table

| Type | Condition | Physical interpretation |
|------|-----------|-------------------------|
| **Dirichlet** | $u = g$ | Fixed value |
| **Neumann** | $\partial u/\partial n = h$ | Fixed flux |
| **Robin** | $\alpha u + \beta \partial u/\partial n = \gamma$ | Mixed (e.g., Newton cooling) |
| **Periodic** | $u(0) = u(L)$, $u'(0)=u'(L)$ | Wrap-around domain |

In [ ]:
# ── Compare BC types on the 1-D steady heat equation u'' = 0 ─────────────────
# Exact solutions are linear: u(x) = A + Bx

L = 1.0
x = np.linspace(0, L, 200)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

# --- Dirichlet–Dirichlet: u(0)=1, u(L)=0  →  u = 1 - x/L -----------------
u_dir = 1 - x / L
axes[0].plot(x, u_dir, 'b', lw=2)
axes[0].scatter([0, L], [1, 0], color='red', zorder=5, s=80, label='BC values')
axes[0].set_title('Dirichlet–Dirichlet\n$u(0)=1,\; u(1)=0$')
axes[0].set_xlabel('x'); axes[0].set_ylabel('u(x)')
axes[0].legend()

# --- Dirichlet–Neumann: u(0)=1, u'(L)=0  →  u = 1 (constant) -------------
u_neu = np.ones_like(x)   # flat profile: slope = 0 everywhere
axes[1].plot(x, u_neu, 'g', lw=2)
axes[1].scatter([0], [1], color='red', zorder=5, s=80, label='Dirichlet BC')
axes[1].annotate("Neumann: $u'(1)=0$\n(insulated)", xy=(L, 1),
                 xytext=(0.6, 1.05), arrowprops=dict(arrowstyle='->'))
axes[1].set_title('Dirichlet–Neumann\n$u(0)=1,\; u\'(1)=0$')
axes[1].set_xlabel('x')
axes[1].legend()

# --- Robin on right: u(0)=1, u'(L)+2u(L)=0  →  u = A + Bx ----------------
# u' + 2u = 0 at x=L: B + 2(A+BL)=0; u(0)=A=1 → B=-2/(1+2L)=-2/3
A_r, B_r = 1.0, -2.0 / (1 + 2 * L)
u_rob = A_r + B_r * x
axes[2].plot(x, u_rob, 'm', lw=2)
axes[2].scatter([0], [1], color='red', zorder=5, s=80, label='Dirichlet BC')
axes[2].set_title('Dirichlet–Robin\n$u(0)=1,\; u\'(1)+2u(1)=0$')
axes[2].set_xlabel('x')
axes[2].legend()

plt.suptitle('Effect of Boundary Condition Type on $u\'\'=0$', fontsize=13)
plt.tight_layout()
plt.show()

---
## 4  Initial Conditions (ICs)

ICs specify the state of the solution **at the starting time** $t = 0$.  
They are required for **time-dependent** (evolution) PDEs.

### 4.1  Parabolic equations — one IC

The heat equation $u_t = \alpha u_{xx}$ is first-order in time, so we need:

$$u(x, 0) = u_0(x) \quad \forall\, x \in \Omega$$

**Example:** a rod initially at temperature $u_0(x) = \sin(\pi x / L)$

### 4.2  Hyperbolic equations — two ICs

The wave equation $u_{tt} = c^2 u_{xx}$ is second-order in time, so we need **both**:

$$u(x, 0) = f(x) \quad \text{(initial displacement)}$$
$$u_t(x, 0) = g(x) \quad \text{(initial velocity)}$$

**Example:** a plucked guitar string — shape $f(x)$ is given, released from rest $g(x)=0$

### 4.3  Intuition: counting conditions

| PDE type | Time order | ICs needed | BCs needed |
|----------|------------|------------|------------|
| Elliptic | — | 0 | On entire $\partial\Omega$ |
| Parabolic | 1st | 1 ($u$ at $t=0$) | On spatial boundary |
| Hyperbolic | 2nd | 2 ($u$ and $u_t$ at $t=0$) | On spatial boundary |

> **Rule of thumb:** the total number of conditions must match the *order* of the PDE in each independent variable.

In [ ]:
# ── Demonstrate IC sensitivity: two different ICs, same heat equation ────────

L = np.pi          # domain [0, π]
alpha = 0.5
x = np.linspace(0, L, 300)
t_plot = [0, 0.1, 0.5, 1.5, 4.0]

def heat_solution(x, t, u0_coeffs, alpha=0.5, L=np.pi):
    """Fourier series solution of u_t=alpha*u_xx, Dirichlet BCs u(0)=u(L)=0.
    u0_coeffs: list of (n, b_n) pairs for the IC u(x,0)=Σ b_n sin(nπx/L)"""
    u = np.zeros_like(x, dtype=float)
    for n, bn in u0_coeffs:
        lam = (n * np.pi / L)**2
        u += bn * np.sin(n * np.pi * x / L) * np.exp(-alpha * lam * t)
    return u

# IC 1: single mode  u₀ = sin(x)
ic1 = [(1, 1.0)]
# IC 2: multi-mode   u₀ ≈ step function (Fourier series)
ic2 = [(n, 4.0 / (n * np.pi) if n % 2 == 1 else 0.0) for n in range(1, 40)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(t_plot)))

for t, col in zip(t_plot, colors):
    ax1.plot(x, heat_solution(x, t, ic1), color=col, lw=1.8, label=f't={t}')
    ax2.plot(x, heat_solution(x, t, ic2), color=col, lw=1.8, label=f't={t}')

ax1.set_title('IC 1: Smooth — $u_0(x)=\\sin(x)$')
ax2.set_title('IC 2: Discontinuous — $u_0(x)\\approx$ step function')
for ax in (ax1, ax2):
    ax.set_xlabel('x'); ax.set_ylabel('u(x,t)')
    ax.legend(fontsize=8)

plt.suptitle('Heat Equation: Role of the Initial Condition', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Hyperbolic: d'Alembert — effect of two different ICs ────────────────────
# Wave eq on R: u(x,t) = [f(x-ct)+f(x+ct)]/2 + 1/(2c) ∫_{x-ct}^{x+ct} g(s)ds

from scipy.integrate import cumulative_trapezoid

c = 1.0
x = np.linspace(-5, 5, 1000)
t_vals = [0, 0.5, 1.0, 2.0]

# Initial displacement: Gaussian pulse
f = lambda s: np.exp(-2 * s**2)

# Case A: released from rest  g(x) = 0
# Case B: initial velocity only  f(x)=0, g(x)=Gaussian

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(t_vals)))

for t, col in zip(t_vals, colors):
    # Case A: u = [f(x-ct)+f(x+ct)]/2
    uA = 0.5 * (f(x - c * t) + f(x + c * t))
    ax1.plot(x, uA, color=col, lw=1.8, label=f't={t}')

    # Case B: u = 1/(2c) ∫_{x-ct}^{x+ct} g(s)ds,  g(s)=exp(-2s²)
    g = lambda s: np.exp(-2 * s**2)
    # Numerical integration over dense grid
    s_dense = np.linspace(-10, 10, 5000)
    G = cumulative_trapezoid(g(s_dense), s_dense, initial=0)
    # Interpolate G at x-ct and x+ct
    G_hi = np.interp(x + c * t, s_dense, G)
    G_lo = np.interp(x - c * t, s_dense, G)
    uB = (G_hi - G_lo) / (2 * c)
    ax2.plot(x, uB, color=col, lw=1.8, label=f't={t}')

ax1.set_title('IC: displacement $f(x)=e^{-2x^2}$, velocity $g=0$\n(two counter-propagating waves)')
ax2.set_title('IC: displacement $f=0$, velocity $g(x)=e^{-2x^2}$\n(energy spreads outward)')
for ax in (ax1, ax2):
    ax.set_xlabel('x'); ax.set_ylabel('u(x,t)')
    ax.legend(fontsize=8)

plt.suptitle("Wave Equation (d'Alembert): Role of Initial Conditions", fontsize=13)
plt.tight_layout()
plt.show()

---
## 5  Well-Posedness: Hadamard's Conditions

A PDE problem is **well-posed** (in the sense of Hadamard) if:

1. **Existence** — A solution exists.
2. **Uniqueness** — The solution is unique.
3. **Stability** — The solution depends continuously on the data (ICs, BCs, forcing).

Violation of any condition → **ill-posed** problem.

### Examples of ill-posedness

| Problem | Issue |
|---------|-------|
| Laplace eq. with a Neumann BC only (no Dirichlet) | Uniqueness fails (solution + const is also a solution) |
| Backward heat equation $u_t = -\alpha u_{xx}$ | Stability fails: high-frequency modes grow exponentially |
| Wave equation with only one IC | Existence or uniqueness may fail |

> **Numerical implication:** Discretising an ill-posed problem will produce an unstable numerical scheme regardless of grid refinement.

In [ ]:
# ── Backward heat equation instability demo ───────────────────────────────────
# u_t = -alpha u_xx  with u(x,0)=sin(x)+ε·sin(Nx)
# Solution blows up for high-freq perturbations

x = np.linspace(0, np.pi, 500)
alpha = 0.1
t = 0.05   # tiny time step — already catastrophic for backward eq.
N = 20     # high-frequency perturbation mode
eps = 0.01

# Backward heat: modal amplification factor = exp(+alpha*(n*pi/L)^2 * t)
amp_base = np.exp(+alpha * (1)**2 * t)       # mode n=1
amp_pert = np.exp(+alpha * (N)**2 * t)       # mode n=N

u_base = np.sin(x)                            # unperturbed
u_pert_0 = np.sin(x) + eps * np.sin(N * x)   # perturbed IC
u_back_base = amp_base * np.sin(x)
u_back_pert = amp_base * np.sin(x) + eps * amp_pert * np.sin(N * x)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(x, u_base, 'b', label='IC: $\\sin(x)$', lw=2)
axes[0].plot(x, u_pert_0, 'r--', label=f'IC: $\\sin(x)+{eps}\\sin({N}x)$', lw=1.5)
axes[0].set_title('Initial Conditions (nearly identical)')
axes[0].legend(); axes[0].set_xlabel('x')

axes[1].plot(x, u_back_base, 'b', label=f'Backward sol., $t={t}$, unperturbed', lw=2)
axes[1].plot(x, u_back_pert, 'r--', label=f'Backward sol., $t={t}$, perturbed', lw=1.5)
axes[1].set_title(f'Backward Heat Eq. at $t={t}$: CATASTROPHIC INSTABILITY')
axes[1].legend(fontsize=8); axes[1].set_xlabel('x')

plt.suptitle('Ill-Posedness: Backward Heat Equation', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Amplification of base mode (n=1):   {amp_base:.4f}×")
print(f"Amplification of perturbation (n={N}): {amp_pert:.2e}×  ← blows up!")

---
## 6  Putting It All Together: A Complete Problem Statement

A well-posed PDE problem consists of:

$$\boxed{\text{PDE} + \text{ICs} + \text{BCs} = \text{Complete Problem}}$$

### Example: heated rod with convective cooling

**Domain:** $x \in [0, L]$, $t > 0$  

**PDE** (heat equation with sink term):
$$\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2} - \beta(u - T_\infty)$$

**IC** (initial temperature distribution):
$$u(x, 0) = T_0 + (T_1 - T_0)\frac{x}{L}$$

**BCs:**
$$u(0, t) = T_0 \quad \text{(Dirichlet — left end fixed)}$$
$$\alpha \frac{\partial u}{\partial x}(L, t) + h\bigl(u(L,t) - T_\infty\bigr) = 0 \quad \text{(Robin — right end convective)}$$

In [ ]:
# ── Numerical solution of the complete example (explicit finite differences) ──

# Parameters
L     = 1.0    # rod length
T0    = 100.0  # left BC temperature
T1    = 200.0  # initial right-end temperature
T_inf = 20.0   # ambient temperature
alpha = 0.01   # thermal diffusivity
beta  = 0.5    # volumetric cooling coefficient
h     = 10.0   # convective heat transfer coefficient

# Grid
N  = 100
dx = L / N
dt = 0.4 * dx**2 / alpha   # stability: r = alpha*dt/dx² < 0.5
x  = np.linspace(0, L, N + 1)

# IC
u = T0 + (T1 - T0) * x / L

t_snapshots = [0.0, 0.5, 2.0, 5.0, 15.0]
snap_idx = 0
results = {}
t = 0.0

if 0.0 in t_snapshots:
    results[0.0] = u.copy()
    snap_idx = 1

max_steps = int(max(t_snapshots) / dt) + 2
for _ in range(max_steps):
    u_new = u.copy()
    # Interior: explicit Euler
    r = alpha * dt / dx**2
    u_new[1:-1] = (u[1:-1]
                   + r * (u[2:] - 2*u[1:-1] + u[:-2])
                   - beta * dt * (u[1:-1] - T_inf))
    # Left BC (Dirichlet)
    u_new[0] = T0
    # Right BC (Robin): α(u_N - u_{N-1})/dx + h(u_N - T_inf) = 0
    # → u_N = (alpha/dx * u_{N-1} + h*T_inf) / (alpha/dx + h)
    u_new[-1] = (alpha / dx * u_new[-2] + h * T_inf) / (alpha / dx + h)

    u = u_new
    t += dt

    if snap_idx < len(t_snapshots) and t >= t_snapshots[snap_idx]:
        results[t_snapshots[snap_idx]] = u.copy()
        snap_idx += 1
    if snap_idx >= len(t_snapshots):
        break

fig, ax = plt.subplots(figsize=(9, 5))
colors = plt.cm.coolwarm(np.linspace(0, 1, len(t_snapshots)))
for (ts, sol), col in zip(results.items(), colors):
    ax.plot(x, sol, color=col, lw=2, label=f't = {ts:.1f} s')

ax.axhline(T_inf, color='k', ls=':', lw=1.2, label=f'$T_\\infty = {T_inf}$ °C')
ax.axvline(0, color='steelblue', ls='--', lw=1, alpha=0.6)
ax.set_xlabel('x (m)')
ax.set_ylabel('Temperature u (°C)')
ax.set_title('Complete Problem: Heat Eq. + Dirichlet/Robin BCs + Linear IC')
ax.legend()
plt.tight_layout()
plt.show()

---
## 7  Classification Flowchart

```
Given a 2nd-order PDE in 2 variables
        Au_xx + Bu_xy + Cu_yy + ... = 0
                        │
           Compute Δ = B² - 4AC
          ┌─────────────┼─────────────┐
        Δ < 0         Δ = 0         Δ > 0
          │             │             │
       ELLIPTIC     PARABOLIC    HYPERBOLIC
          │             │             │
     No time var    One time-like  Two time-like
     Needs BCs on   direction      directions
     full boundary  IC + BCs       2 ICs + BCs
          │             │             │
      Laplace       Heat eq.      Wave eq.
      Poisson       Diffusion     Acoustics
      Steady flow   Finance       Elasticity
```

---
## 8  Summary

| Concept | Key takeaway |
|---------|-------------|
| **PDE classification** | Discriminant $B^2-4AC$ determines elliptic/parabolic/hyperbolic character |
| **Dirichlet BC** | Fix $u$ on boundary — strongest constraint |
| **Neumann BC** | Fix flux $\partial u/\partial n$ — allows sliding of value |
| **Robin BC** | Mix of value and flux — models convection |
| **Initial Conditions** | 1 IC for parabolic; 2 ICs for hyperbolic; 0 for elliptic |
| **Well-posedness** | Existence + Uniqueness + Stability (Hadamard) |
| **Ill-posedness risk** | Backward diffusion, missing BCs/ICs cause blow-up or non-uniqueness |

---
## 9  Exercises

1. **Classification:** Determine whether each of the following is elliptic, parabolic, or hyperbolic:
   - $u_{xx} + 4u_{xy} + 4u_{yy} = 0$
   - $u_{xx} - u_{yy} = \sin x$
   - $3u_{xx} + 2u_{xy} + 3u_{yy} = 0$

2. **BC types:** For a vibrating membrane (2-D wave equation) clamped at its edges, what type of BC is applied?

3. **Well-posedness:** The Neumann problem for the Laplace equation $\nabla^2 u = 0$ on a bounded domain with $\partial u / \partial n = h$ on all boundaries is only solvable if $\oint_{\partial\Omega} h\,ds = 0$. Why? What physical law does this reflect?

4. **Coding challenge:** Modify the finite-difference code in Section 6 to use a **Neumann BC** on the right end ($u_x(L,t)=0$, insulated). How does the steady-state solution change?

5. **Backward heat equation:** Starting from the code in Section 5, run the forward heat equation for $t=1$, then use the output as the IC for the backward equation for $t=1$. Do you recover the original IC? What happens when you add a tiny random perturbation?

---
## References

- Strauss, W. A. *Partial Differential Equations: An Introduction* (2nd ed., Wiley, 2008)
- Evans, L. C. *Partial Differential Equations* (AMS, 2010)
- LeVeque, R. J. *Finite Difference Methods for Ordinary and Partial Differential Equations* (SIAM, 2007)
- Trefethen, L. N. *Spectral Methods in MATLAB* (SIAM, 2000)